# Notebook 6 — Envío de predicciones al PLC Beckhoff (ADS)
## Del clasificador a la acción del brazo

Último paso de la **Parte 2 (Visión)**. Aquí el notebook **solo envía datos** al IPC
Beckhoff por ADS; el **HMI con las lámparas y la lógica del brazo se programan en
TwinCAT** (eso lo haces tú en TwinCAT, no en el notebook).

Avanzamos de lo simple a lo completo:

| Etapa | Qué se hace |
|------|-------------|
| 1. Setup | Imports y configuración (AMS Net ID, variables) |
| 2. Conexión | Abrir la conexión ADS con el IPC |
| **3. Envío de booleanos (prueba)** | Prender/apagar `bRojo/bAzul/bAmarillo` con botones y verlos en tus **lámparas de TwinCAT** |
| 4. Predicción → envío | Clasificar **una** imagen y enviar ese resultado al PLC |
| 5. (Extra) Cámara en vivo | Clasificar la cámara en tiempo real y enviar continuamente |

> **Antes de empezar (en TwinCAT):**
> 1. Crea un proyecto TwinCAT y declara en una GVL tres variables `BOOL`:
>    `bRojo`, `bAzul`, `bAmarillo`.
> 2. Arma un HMI sencillo con una **lámpara por color** ligada a cada BOOL.
> 3. Activa la configuración (Login + Run) y anota el **AMS Net ID** del IPC.
>
> Sin hardware/TwinCAT, el notebook funciona en **modo simulación**: imprime lo que
> enviaría, para que puedas probar la lógica.

> ⚠️ **Código ADS de referencia.** Las celdas que usan `pyads` se entregan **como
> referencia/sugerencia** para orientar la integración; **no han sido probadas con tu
> hardware** y pueden requerir ajustes (AMS Net ID, ruta/route ADS, nombres de variables,
> permisos del *router*) o incluso fallar. **Completar, adaptar y validar la comunicación
> con el PLC es parte del trabajo de la pareja.** Consulta la documentación oficial de la
> librería:
>
> - **pyads (docs):** https://pyads.readthedocs.io
> - **pyads (repositorio):** https://github.com/stlehmann/pyads

---
## 1. Setup y configuración

In [ ]:
import os
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '-1')
import os, glob, time
import io as _io
import json as _json
from collections import deque
import numpy as np
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
tf.get_logger().setLevel('ERROR')
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import ipywidgets as widgets
from IPython.display import display, clear_output

try:
    import pyads
    PYADS_OK = True
except ImportError:
    PYADS_OK = False

# ───────── CONFIGURACIÓN (ajusta a tu equipo) ─────────
AMS_NET_ID = '5.80.201.232.1.1'   # <-- AMS Net ID de TU IPC Beckhoff
ADS_PORT   = 851                  # 851 = TwinCAT 3 Runtime 1
UMBRAL_CONFIANZA = 0.85           # confianza mínima para activar un color

# Nombres EXACTOS de tus variables BOOL en TwinCAT.
# Si tu GVL no se llama 'VARIABLES', cambia el prefijo (p.ej. 'GVL.bRojo').
VARIABLES = {
    'rojo':     'VARIABLES.bRojo',
    'azul':     'VARIABLES.bAzul',
    'amarillo': 'VARIABLES.bAmarillo',
}

cfg = {'model': None, 'class_names': [], 'img_size': 128, 'preprocessing': 'rescale'}

def _detectar_preprocesamiento(model):
    for layer in model.layers:
        if 'mobilenet' in layer.name.lower():
            return 'mobilenet'
    return 'rescale'

print('TensorFlow', tf.__version__, '| pyads:', 'OK' if PYADS_OK else 'no instalado (simulación)')

---
## 2. Conexión con el PLC

Abre la conexión ADS con el IPC. Requisitos:
- **TwinCAT en Run** y el **AMS Net ID** correcto (celda anterior).
- Una **ruta ADS** entre tu PC y el target. Si falta, verás `Missing ADS routes (7)`:
  créala en **TwinCAT (bandeja) → Router → Edit Routes → Add Route**
  (o con `pyads.add_route_to_plc(...)`).

`conectar_plc()` hace una lectura de prueba (`read_state`) para verificar la ruta; si falla
(o no hay `pyads`), el notebook sigue en **modo simulación** sin detenerse.

In [ ]:
# Código de referencia (NO probado con tu hardware) — adáptalo según tu setup.
# API pyads: https://pyads.readthedocs.io  ·  repo: https://github.com/stlehmann/pyads
plc = {'conn': None, 'connected': False}

def conectar_plc(ams=AMS_NET_ID, port=ADS_PORT):
    if not PYADS_OK:
        plc['connected'] = False
        print('pyads no instalado  ->  MODO SIMULACIÓN'); return False
    try:
        conn = pyads.Connection(ams, port)
        conn.open()
        conn.read_state()   # verifica que exista la RUTA ADS (si falta, lanza ADSError)
        plc['conn'] = conn; plc['connected'] = True
        print(f'✔ Conectado a {ams}:{port}')
        return True
    except Exception as e:
        try:
            if plc.get('conn') is not None:
                plc['conn'].close()
        except Exception:
            pass
        plc['conn'] = None; plc['connected'] = False
        print(f'No se pudo conectar / sin ruta ADS ({e})  ->  MODO SIMULACIÓN')
        print('  Si dice "Missing ADS routes (7)": falta la RUTA ADS entre tu PC y el target.')
        print('  Solucion: TwinCAT (systray) > Router > Edit Routes > Add Route,')
        print('  o usar pyads.add_route_to_plc(...). Doc: https://pyads.readthedocs.io')
        try:
            pyads.open_port()
            print('  (info) Tu AMS Net ID LOCAL es:', pyads.get_local_address().netid)
            pyads.close_port()
            print('        Si el PLC corre en ESTA misma PC, usa ese Net ID arriba.')
        except Exception:
            pass
        return False

def desconectar_plc():
    if plc['conn'] is not None and plc['connected']:
        try: plc['conn'].close()
        except Exception: pass
    plc['conn'] = None; plc['connected'] = False

conectar_plc()

---
## 3. Prueba: enviar booleanos manualmente

Esta es la prueba clave: **¿puedo enviar datos al PLC?** Pulsa cada botón y observa
cómo se prende/apaga la **lámpara correspondiente en tu HMI de TwinCAT**. Cada color
activa su BOOL y apaga los otros dos (exclusión mutua); *Apagar todo* pone los tres en
`False`.

In [ ]:
def escribir_bool(var, valor):
    """Escribe un BOOL en el PLC (o lo reporta en simulación)."""
    if not plc['connected']:
        return valor
    try:
        plc['conn'].write_by_name(var, bool(valor), pyads.PLCTYPE_BOOL)
    except Exception as e:
        print(f'  ⚠ Error ADS al escribir {var}: {e}  ->  paso a MODO SIMULACIÓN')
        plc['connected'] = False   # evita repetir el error en cada variable/frame
    return valor

def enviar_color(color):
    """Activa el BOOL del color indicado y apaga los demás. color=None apaga todo."""
    estado = {}
    for c, var in VARIABLES.items():
        val = (c == color)
        escribir_bool(var, val)
        estado[var] = val
    modo = 'PLC' if plc['connected'] else 'SIM'
    activos = [v for v, on in estado.items() if on] or ['(ninguno)']
    print(f'[{modo}] activo: {activos[0]}   ', {v: estado[v] for v in estado})
    return estado

def apagar_todo():
    return enviar_color(None)

# --- Botones de prueba ---
_out_bools = widgets.Output()
_botones = []
_estilos = {'rojo': 'danger', 'azul': 'primary', 'amarillo': 'warning'}
for _c in VARIABLES:
    _b = widgets.Button(description=f'{_c.capitalize()} ON', button_style=_estilos.get(_c, 'info'))
    def _mk(col):
        def _h(_):
            with _out_bools:
                clear_output(wait=True); enviar_color(col)
        return _h
    _b.on_click(_mk(_c)); _botones.append(_b)
_b_off = widgets.Button(description='Apagar todo', icon='power-off')
def _off(_):
    with _out_bools:
        clear_output(wait=True); apagar_todo()
_b_off.on_click(_off); _botones.append(_b_off)
display(widgets.HBox(_botones)); display(_out_bools)

---
## 4. Predicción sencilla → envío al PLC

Ahora una predicción real: carga el modelo de la Parte 2, clasifica **una** imagen
(súbela o toma una del dataset) y pulsa **Enviar al PLC** para activar el BOOL del
color detectado. Si la confianza no supera el umbral, no se envía nada.

In [ ]:
# --- Cargar modelo entrenado (busca en modelos/) ---
_modelos = {}
for _d in ['modelos', '../modelos', '.']:
    if not os.path.isdir(_d):
        continue
    for _h5 in glob.glob(os.path.join(_d, '*.h5')) + glob.glob(os.path.join(_d, '*.keras')):
        _jp = _h5.rsplit('.', 1)[0] + '.json'
        _meta = _json.load(open(_jp)) if os.path.exists(_jp) else {}
        _va = _meta.get('val_accuracy')
        _lbl = os.path.basename(_h5) + (f'  (val {_va:.0%})' if _va is not None else '')
        _modelos[_lbl] = {'path': _h5, 'meta': _meta}

def _cargar_modelo(path):
    meta = {}; jp = path.rsplit('.', 1)[0] + '.json'
    if os.path.exists(jp): meta = _json.load(open(jp))
    model = load_model(path, compile=False)
    cfg['model'] = model
    cfg['img_size'] = meta.get('img_size', model.input_shape[1])
    cfg['preprocessing'] = meta.get('preprocessing', _detectar_preprocesamiento(model))
    cfg['class_names'] = meta.get('class_names', [f'Clase {i}' for i in range(model.output_shape[-1])])
    print('Modelo:', os.path.basename(path), '| clases', cfg['class_names'], '| prep', cfg['preprocessing'])

if _modelos:
    _dd = widgets.Dropdown(options=list(_modelos), description='Modelo:',
                           layout=widgets.Layout(width='100%'), style={'description_width': 'initial'})
    _bcarga = widgets.Button(description='Cargar', button_style='success', icon='check')
    _ocarga = widgets.Output()
    def _hc(_):
        with _ocarga:
            clear_output(wait=True); _cargar_modelo(_modelos[_dd.value]['path'])
    _bcarga.on_click(_hc)
    display(widgets.VBox([_dd, _bcarga, _ocarga]))
else:
    print('No se encontraron modelos. Entrena uno en notebook4_transfer_learning.ipynb')

In [ ]:
ultima = {'color': None, 'conf': 0.0}
_img_pred = widgets.Image(format='png')
_out_envio = widgets.Output()

def _prep_arr(img):
    arr = img_to_array(img).astype('float32')
    if cfg['preprocessing'] == 'mobilenet':
        from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
        return preprocess_input(arr)
    return arr / 255.0

def _predecir(path):
    if cfg['model'] is None:
        with _out_envio:
            clear_output(wait=True); print('Carga un modelo primero (arriba).')
        return
    sz = cfg['img_size']; names = cfg['class_names']
    img = load_img(path, target_size=(sz, sz))
    pred = cfg['model'].predict(np.expand_dims(_prep_arr(img), 0), verbose=0)[0]
    idx = int(np.argmax(pred)); clase = names[idx]; conf = float(pred[idx])
    ultima['color'] = clase if (conf >= UMBRAL_CONFIANZA and clase in VARIABLES) else None
    ultima['conf'] = conf
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.6), gridspec_kw={'width_ratios': [1, 1.2]})
    a1.imshow(img); a1.axis('off'); a1.set_title(f'Pred: {clase}  ({conf:.0%})', fontweight='bold')
    cols = ['#2ecc71' if n == clase else '#bdc3c7' for n in names]
    a2.barh(names, pred, color=cols); a2.set_xlim(0, 1); a2.set_xlabel('Confianza')
    plt.tight_layout(); buf = _io.BytesIO(); fig.savefig(buf, format='png', bbox_inches='tight', dpi=90)
    plt.close(fig); _img_pred.value = buf.getvalue()
    with _out_envio:
        clear_output(wait=True)
        if ultima['color']:
            print(f"Listo para enviar: {ultima['color']}  ->  {VARIABLES[ultima['color']]} = True")
        else:
            print(f"Sin acción: clase '{clase}' no mapeada o confianza < {UMBRAL_CONFIANZA:.0%}.")

_val_imgs = []
for _ext in ('*.jpg', '*.jpeg', '*.png'):
    for _dd2 in ('data/tapitas', '../data/tapitas'):
        _val_imgs += glob.glob(os.path.join(_dd2, '*', _ext))

_b_rand = widgets.Button(description='Imagen aleatoria', icon='random')
_up = widgets.FileUpload(accept='image/*', multiple=False, description='Subir imagen')
_b_send = widgets.Button(description='Enviar al PLC', button_style='success', icon='paper-plane')

def _on_rand(_=None):
    if not _val_imgs:
        with _out_envio:
            clear_output(wait=True); print('No hay imágenes en data/tapitas.')
        return
    import random as _r; _predecir(_r.choice(_val_imgs))

def _on_up(change):
    if not _up.value: return
    item = list(_up.value.values())[0] if isinstance(_up.value, dict) else _up.value[0]
    content = item['content'] if isinstance(item, dict) else item.content
    p = '_pred_nb6.jpg'
    with open(p, 'wb') as f: f.write(content)
    _predecir(p)

def _on_send(_):
    with _out_envio:
        if ultima['color']:
            enviar_color(ultima['color'])
        else:
            print('Nada que enviar (predice algo con confianza suficiente).')

_b_rand.on_click(_on_rand); _up.observe(_on_up, names='value'); _b_send.on_click(_on_send)
display(widgets.HBox([_b_rand, _up, _b_send])); display(_img_pred); display(_out_envio)

---
## 5. (Extra) Cámara en vivo → PLC

Versión continua: clasifica la cámara en tiempo real (con suavizado anti-parpadeo) y
envía el color detectado al PLC en cada frame. **`Q`** para salir (al salir apaga los
tres BOOLs).

> En **WSL2** no hay cámara USB: ejecuta este notebook desde **Windows**.

In [ ]:
CAMERA_INDEX = 0
SMOOTHING_WINDOW = 5

def _es_wsl():
    try: return 'microsoft' in open('/proc/version').read().lower()
    except Exception: return False

def _draw(frame, clase, conf, enviado, fps):
    h, w = frame.shape[:2]
    ov = frame.copy(); cv2.rectangle(ov, (0, 0), (w, 56), (30, 30, 30), -1)
    cv2.addWeighted(ov, 0.85, frame, 0.15, 0, frame)
    activo = enviado is not None
    col = (0, 230, 0) if activo else (150, 150, 150)
    txt = f'{clase}  {conf*100:.0f}%' if activo else f'{clase}?  {conf*100:.0f}%'
    cv2.putText(frame, txt, (14, 38), cv2.FONT_HERSHEY_SIMPLEX, 1.0, col, 2, cv2.LINE_AA)
    modo = 'PLC' if plc['connected'] else 'SIM'
    cv2.putText(frame, f'{modo} {fps:.0f}fps', (w - 150, 36), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (180, 180, 180), 1, cv2.LINE_AA)
    cv2.putText(frame, 'Q = salir', (10, h - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (140, 140, 140), 1, cv2.LINE_AA)
    return frame

if cfg['model'] is None:
    print('Carga un modelo primero (paso 4).')
elif _es_wsl():
    print('WSL2 no accede a la cámara USB. Ejecuta este notebook desde Windows.')
else:
    cap = cv2.VideoCapture(CAMERA_INDEX)
    if not cap.isOpened():
        print(f'No se pudo abrir la cámara (índice {CAMERA_INDEX}).')
    else:
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640); cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
        sz = cfg['img_size']; names = cfg['class_names']; prep = cfg['preprocessing']
        buf = deque(maxlen=SMOOTHING_WINDOW); prev = 0
        print('Cámara abierta. Modo:', 'PLC' if plc['connected'] else 'SIMULACIÓN', "  'Q' para salir.")
        try:
            while True:
                ok, frame = cap.read()
                if not ok: break
                now = time.time(); fps = 1 / (now - prev) if prev else 0; prev = now
                img = cv2.cvtColor(cv2.resize(frame, (sz, sz)), cv2.COLOR_BGR2RGB)
                pred = cfg['model'].predict(np.expand_dims(_prep_arr(img), 0), verbose=0)[0]
                buf.append(pred); p = np.mean(buf, axis=0)
                idx = int(np.argmax(p)); clase = names[idx]; conf = float(p[idx])
                color = clase if (conf >= UMBRAL_CONFIANZA and clase in VARIABLES) else None
                enviar_color(color)
                frame = _draw(frame, clase, conf, color, fps)
                cv2.imshow('NB6 - Inferencia -> Beckhoff', frame)
                if cv2.waitKey(1) & 0xFF == ord('q'): break
        finally:
            apagar_todo(); cap.release(); cv2.destroyAllWindows()
            print('Cámara cerrada y BOOLs en False.')